# LesionGraph-Net — Paper-guided implementation
**Paper:** *LesionGraph-Net: A clinically motivated Graph Neural Network for accurate skin cancer diagnosis using lesion-focused features* (2026)

This Colab notebook implements the main pipeline: ISIC 2019 label mapping → hair removal → LAB-CLAHE → K-means ROI segmentation → nine lesion features → train-only ANOVA/Chi-square feature selection → Pearson graph → GCN + ConvLSTM-style layer-depth fusion → evaluation.

> **Important:** This is a reproducible research implementation, not a clinically validated diagnostic system. The paper has some architectural/reporting ambiguities (including its global feature graph versus its later per-image 8-node description). The notebook makes the per-image 7-feature + `Class_C` node design explicit, estimates graph edges from training data only, and documents the ConvLSTM interpretation in code. Do not expect the paper's reported accuracy to reproduce automatically; dataset curation and split details matter.

In [ ]:
# 1. Install dependencies (run in Google Colab)
!pip -q install opencv-contrib-python scikit-learn pandas matplotlib seaborn tqdm

In [ ]:
# 2. Imports and configuration
import os, glob, random, warnings
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import f_classif, SelectKBest, chi2
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             matthews_corrcoef, roc_curve, auc)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 256
BATCH_SIZE = 64
EPOCHS = 250
LR = 0.01
DROPOUT = 0.30
EDGE_THRESHOLD = 0.20

print("Device:", DEVICE)

## 3. Dataset setup

For the paper's ISIC 2019 experiment, download the ISIC 2019 images and the `ISIC_2019_Training_GroundTruth.csv` file from the official ISIC archive. Set the paths below.

The paper's binary mapping is:
- **Malignant (1):** MEL, BCC, SCC, AK
- **Benign (0):** NV, BKL, DF, VASC
- UNK is excluded.

The paper reports a curated subset of 3,297 images (1,800 malignant / 1,497 benign), not the complete unfiltered ISIC 2019 archive. To reproduce that exact subset, you must apply its lesion-ID, duplicate, metadata and sampling filters. This notebook performs the stated diagnosis mapping and stratified split; it does not claim to recreate the authors' private curation decisions.

In [ ]:
# Update these paths for your Google Drive / Colab dataset
IMAGE_DIR = "/content/ISIC_2019_Training_Input"
GROUND_TRUTH_CSV = "/content/ISIC_2019_Training_GroundTruth.csv"

# Optional: mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

def build_isic_dataframe(image_dir, ground_truth_csv):
    gt = pd.read_csv(ground_truth_csv)
    # ISIC 2019 one-hot columns
    diagnosis_cols = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC", "UNK"]
    missing = [c for c in diagnosis_cols if c not in gt.columns]
    if missing:
        raise ValueError(f"Ground truth CSV is missing columns: {missing}")
    malignant_cols = ["MEL", "BCC", "SCC", "AK"]
    benign_cols = ["NV", "BKL", "DF", "VASC"]

    rows = []
    for _, row in gt.iterrows():
        image_id = str(row["image"])
        if row[malignant_cols].sum() == 1:
            label = 1
        elif row[benign_cols].sum() == 1:
            label = 0
        else:
            continue
        candidates = [os.path.join(image_dir, image_id + ext) for ext in [".jpg", ".jpeg", ".png"]]
        image_path = next((p for p in candidates if os.path.exists(p)), None)
        if image_path:
            rows.append({"image": image_id, "path": image_path, "label": label})
    df = pd.DataFrame(rows).drop_duplicates("image").reset_index(drop=True)
    return df

df = build_isic_dataframe(IMAGE_DIR, GROUND_TRUTH_CSV)
print(df.shape)
print(df["label"].value_counts().rename(index={0:"Benign", 1:"Malignant"}))
df.head()

## 4. Preprocessing and K-means segmentation

Paper settings:
- Black-hat hair filtering with a 17×17 elliptical kernel, threshold 10, dilation 3×3 and Telea inpainting (radius 3).
- CLAHE on the L channel in LAB (clip limit 2.0, tile grid 8×8).
- K-means in LAB with K=2; the cluster with lower mean L is treated as lesion foreground.
- Morphological opening/closing and largest connected component.

K-means is an approximate coarse ROI extractor. It can fail on low-contrast, very dark or atypical lesions, so inspect the masks before trusting downstream features.

In [ ]:
def remove_hair_bhat(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, hair_mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    hair_mask = cv2.dilate(hair_mask, np.ones((3,3), np.uint8), iterations=1)
    return cv2.inpaint(bgr, hair_mask, 3, cv2.INPAINT_TELEA)

def enhance_clahe_lab(bgr):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l2 = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l2, a, b]), cv2.COLOR_LAB2BGR)

def segment_kmeans_lab(bgr, k=2):
    h, w = bgr.shape[:2]
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    pixels = lab.reshape((-1, 3)).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    cv2.setRNGSeed(SEED)
    _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
    labels = labels.reshape(h, w)
    lesion_cluster = int(np.argmin(centers[:, 0]))  # darkest mean L cluster
    mask = (labels == lesion_cluster).astype(np.uint8) * 255

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    n, cc, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = np.where(cc == largest, 255, 0).astype(np.uint8)
    return mask

def preprocess_image(path):
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise ValueError(f"Could not read image: {path}")
    bgr = cv2.resize(bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    clean = remove_hair_bhat(bgr)
    enhanced = enhance_clahe_lab(clean)
    mask = segment_kmeans_lab(enhanced, k=2)
    return bgr, enhanced, mask

# Visual sanity check on one sample
orig, enhanced, mask = preprocess_image(df.iloc[0]["path"])
fig, ax = plt.subplots(1,3, figsize=(12,4))
ax[0].imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)); ax[0].set_title("Original")
ax[1].imshow(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)); ax[1].set_title("Preprocessed")
ax[2].imshow(mask, cmap="gray"); ax[2].set_title("K-means mask")
for a in ax: a.axis("off")
plt.show()

## 5. Extract the nine computerized features

The paper's nine features are: solidity/convex-hull ratio, SIFT keypoint density, Harris corner density, elliptic ratio, circularity, width-height ratio, asymmetry score, color variation and compactness index.

Definitions follow the paper's Table 4. For SIFT/Harris, points are counted in a narrow band around the segmented lesion boundary and normalized by lesion area. Elliptic ratio is fitted-ellipse area divided by lesion area. Asymmetry is measured by splitting the mask through its centroid along its principal axis.

In [ ]:
FEATURE_NAMES = [
    "solidity", "sift_density", "harris_density", "elliptic_ratio",
    "circularity", "wh_ratio", "asymmetry_score", "color_variation",
    "compactness_index"
]

def extract_features(bgr, enhanced, mask):
    binary = (mask > 0).astype(np.uint8)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.zeros(len(FEATURE_NAMES), dtype=np.float32)

    contour = max(contours, key=cv2.contourArea)
    area = float(cv2.contourArea(contour))
    perimeter = float(cv2.arcLength(contour, True))
    if area < 10 or perimeter < 1:
        return np.zeros(len(FEATURE_NAMES), dtype=np.float32)

    hull = cv2.convexHull(contour)
    hull_area = max(float(cv2.contourArea(hull)), 1.0)
    solidity = area / hull_area

    x, y, w, h = cv2.boundingRect(contour)
    wh_ratio = float(w) / max(float(h), 1.0)
    circularity = (4.0 * np.pi * area) / max(perimeter**2, 1e-8)
    compactness = (perimeter**2) / max(4.0 * np.pi * area, 1e-8)

    # Best-fit ellipse area / lesion contour area
    if len(contour) >= 5:
        (_, _), (ma, mi), _ = cv2.fitEllipse(contour)
        ellipse_area = np.pi * (float(ma)/2.0) * (float(mi)/2.0)
        elliptic_ratio = ellipse_area / area
    else:
        elliptic_ratio = 0.0

    # Boundary band for SIFT/Harris keypoints
    boundary = cv2.morphologyEx(mask, cv2.MORPH_GRADIENT,
                                cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
    gray = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY)

    sift = cv2.SIFT_create()
    keypoints = sift.detect(gray, boundary)
    sift_density = len(keypoints) / area

    harris = cv2.cornerHarris(np.float32(gray), 2, 3, 0.04)
    harris = cv2.dilate(harris, None)
    boundary_bool = boundary > 0
    if np.any(boundary_bool):
        threshold = np.percentile(harris[boundary_bool], 90)
        harris_count = int(np.sum((harris > threshold) & boundary_bool))
    else:
        harris_count = 0
    harris_density = harris_count / area

    # Principal-axis asymmetry: rotate lesion's major axis to horizontal,
    # then compare the two halves around the centroid.
    ys, xs = np.where(binary > 0)
    coords = np.column_stack([xs, ys]).astype(np.float32)
    if len(coords) >= 3:
        centered = coords - coords.mean(axis=0, keepdims=True)
        cov = np.cov(centered.T)
        vals, vecs = np.linalg.eigh(cov)
        major = vecs[:, np.argmax(vals)]
        angle = np.degrees(np.arctan2(major[1], major[0]))
        center = (float(xs.mean()), float(ys.mean()))
        rot = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(binary, rot, (binary.shape[1], binary.shape[0]))
        cx = int(round(center[0]))
        left = int(rotated[:, :cx].sum())
        right = int(rotated[:, cx:].sum())
        asymmetry = abs(left-right) / max(left+right, 1)
    else:
        asymmetry = 0.0

    # Standard deviation of grayscale intensities within ROI
    roi_pixels = gray[binary > 0].astype(np.float32)
    color_variation = float(np.std(roi_pixels)) if roi_pixels.size else 0.0

    return np.array([
        solidity, sift_density, harris_density, elliptic_ratio,
        circularity, wh_ratio, asymmetry, color_variation, compactness
    ], dtype=np.float32)

# Feature extraction can take time. Cache the results to avoid rerunning.
CACHE_CSV = "/content/lesion_features_cache.csv"

def create_feature_table(dataframe, cache_path=CACHE_CSV):
    if os.path.exists(cache_path):
        cached = pd.read_csv(cache_path)
        if len(cached) == len(dataframe) and cached["path"].tolist() == dataframe["path"].tolist():
            return cached
    records = []
    for row in tqdm(dataframe.itertuples(index=False), total=len(dataframe)):
        try:
            _, enhanced, mask = preprocess_image(row.path)
            feats = extract_features(cv2.imread(row.path), enhanced, mask)
            rec = {"image": row.image, "path": row.path, "label": int(row.label)}
            rec.update(dict(zip(FEATURE_NAMES, feats.tolist())))
            records.append(rec)
        except Exception as e:
            print("Skipped:", row.path, "|", e)
    out = pd.DataFrame(records)
    out.to_csv(cache_path, index=False)
    return out

features_df = create_feature_table(df)
print("Feature table:", features_df.shape)
features_df.head()

## 6. Split data and select features without leakage

Use stratified 80/10/10 train/validation/test split (seed 42). Fit MinMax scaling and feature selection using **training data only**. The paper's final seven are SIFT, asymmetry, convex-hull ratio/solidity, circularity, color variation, elliptic ratio and Harris density. The code calculates ANOVA and Chi-square rankings on the training split and reports their agreement. To align with the paper's reported final set, the seven named features are retained; all transforms are fitted on training data only.

In [ ]:
# Keep a stable, stratified 80/10/10 split
train_df, temp_df = train_test_split(
    features_df, test_size=0.20, random_state=SEED, stratify=features_df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)

X_train_raw = train_df[FEATURE_NAMES].values.astype(np.float32)
y_train = train_df["label"].values.astype(np.int64)
X_val_raw = val_df[FEATURE_NAMES].values.astype(np.float32)
y_val = val_df["label"].values.astype(np.int64)
X_test_raw = test_df[FEATURE_NAMES].values.astype(np.float32)
y_test = test_df["label"].values.astype(np.int64)

# Paper's seven retained biomarkers
SELECTED_FEATURES = [
    "solidity", "sift_density", "harris_density", "elliptic_ratio",
    "circularity", "asymmetry_score", "color_variation"
]
# Order matches the seven features used in this implementation.
selected_idx = [FEATURE_NAMES.index(c) for c in SELECTED_FEATURES]

# Report train-only ANOVA and chi-square ranks
minmax_all = MinMaxScaler()
X_train_mm_all = minmax_all.fit_transform(X_train_raw)
anova_scores, _ = f_classif(X_train_raw, y_train)
chi_selector_all = SelectKBest(score_func=chi2, k="all").fit(X_train_mm_all, y_train)
ranking = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "anova_F": anova_scores,
    "chi2": chi_selector_all.scores_
}).sort_values("anova_F", ascending=False)
display(ranking)

# Fit transforms only on train data
feature_scaler = StandardScaler()
X_train = feature_scaler.fit_transform(X_train_raw[:, selected_idx]).astype(np.float32)
X_val = feature_scaler.transform(X_val_raw[:, selected_idx]).astype(np.float32)
X_test = feature_scaler.transform(X_test_raw[:, selected_idx]).astype(np.float32)

print("Selected features:", SELECTED_FEATURES)
print("Split sizes:", len(X_train), len(X_val), len(X_test))

## 7. Build the Pearson correlation graph

The seven clinical feature nodes are connected using Pearson correlation calculated from the **training split only**. Retain an undirected edge when `abs(r) > 0.2`; use `abs(r)` as the edge weight. Add one `Class_C` placeholder node connected to the seven feature nodes. At inference its input value is zero; the classifier reads the learned embedding of this node. The true class label is used only for the loss, never as a node input.

In [ ]:
def build_adjacency(X_train_features, threshold=EDGE_THRESHOLD):
    corr = np.corrcoef(X_train_features, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    n = corr.shape[0]
    A = np.zeros((n+1, n+1), dtype=np.float32)  # + Class_C node
    W = np.zeros_like(A)

    for i in range(n):
        for j in range(i+1, n):
            if abs(corr[i,j]) > threshold:
                A[i,j] = A[j,i] = 1.0
                W[i,j] = W[j,i] = abs(corr[i,j])

    # Class_C receives messages from every clinical feature node.
    class_node = n
    for i in range(n):
        A[i,class_node] = A[class_node,i] = 1.0
        W[i,class_node] = W[class_node,i] = 1.0

    # Self loops and symmetric GCN normalization
    A_tilde = A + np.eye(n+1, dtype=np.float32)
    W_tilde = W + np.eye(n+1, dtype=np.float32)
    degree = W_tilde.sum(axis=1)
    inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(degree, 1e-8)))
    A_norm = inv_sqrt @ W_tilde @ inv_sqrt
    return corr, A, W, A_norm.astype(np.float32)

corr, adjacency, edge_weights, A_norm = build_adjacency(X_train)
print("Adjacency matrix shape:", adjacency.shape)
print("Undirected feature edges:", int(np.triu(adjacency[:7,:7], 1).sum()))
plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="coolwarm", xticklabels=SELECTED_FEATURES,
            yticklabels=SELECTED_FEATURES, vmin=-1, vmax=1)
plt.xticks(rotation=60, ha="right"); plt.title("Training-only Pearson correlations")
plt.tight_layout(); plt.show()

## 8. LesionGraph-Net: GCN + ConvLSTM1D fusion

Implementation choices based on the paper:
- 8 nodes per image: 7 feature nodes + `Class_C`.
- Scalar node inputs are projected to 64 dimensions.
- Three weighted GCN message-passing layers, hidden size 64.
- A 1D convolutional LSTM fuses the sequence of node embeddings produced by the three GCN depths (the paper describes this as pseudo-temporal layer-wise refinement).
- Dropout 0.3, batch normalization, ELU/SoftPlus-style nonlinear processing and a two-class output.
- SGD, learning rate 0.01, batch size 64, 250 epochs.

The paper's text and figure do not specify every tensor operation of the ConvLSTM/FFN unambiguously. This is an explicit implementation of the described concept, not the authors' original source code.

In [ ]:
class ConvLSTMCell1D(nn.Module):
    def __init__(self, input_channels, hidden_channels, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.hidden_channels = hidden_channels
        self.gates = nn.Conv1d(input_channels + hidden_channels,
                               4 * hidden_channels,
                               kernel_size=kernel_size, padding=padding)

    def forward(self, x, state):
        h, c = state
        gates = self.gates(torch.cat([x, h], dim=1))
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f*c + i*g
        h_next = o*torch.tanh(c_next)
        return h_next, c_next

class ConvLSTM1D(nn.Module):
    # Input: [batch, sequence_length, channels, nodes]
    def __init__(self, channels, hidden_channels=None):
        super().__init__()
        hidden_channels = hidden_channels or channels
        self.cell = ConvLSTMCell1D(channels, hidden_channels, kernel_size=3)

    def forward(self, sequence):
        b, steps, channels, nodes = sequence.shape
        h = torch.zeros(b, self.cell.hidden_channels, nodes, device=sequence.device)
        c = torch.zeros_like(h)
        for t in range(steps):
            h, c = self.cell(sequence[:, t], (h, c))
        return h  # [B, hidden_channels, nodes]

class WeightedGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.0):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.bn = nn.BatchNorm1d(out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, adj_norm):
        # x: [B, nodes, features], adj_norm: [nodes, nodes]
        support = self.linear(x)
        out = torch.matmul(adj_norm.unsqueeze(0), support)
        b, n, c = out.shape
        out = self.bn(out.reshape(b*n, c)).reshape(b, n, c)
        return self.dropout(F.elu(out))

class LesionGraphNet(nn.Module):
    def __init__(self, num_features=7, hidden=64, dropout=0.3):
        super().__init__()
        self.pre = nn.Sequential(
            nn.Linear(1, hidden), nn.BatchNorm1d(hidden), nn.ELU()
        )
        self.gcn1 = WeightedGCNLayer(hidden, hidden, dropout=0.0)
        self.gcn2 = WeightedGCNLayer(hidden, hidden, dropout=dropout)
        self.gcn3 = WeightedGCNLayer(hidden, hidden, dropout=0.0)
        self.fusion = ConvLSTM1D(hidden, hidden)
        self.post = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2)
        )

    def forward(self, feature_values, adj_norm):
        # feature_values: [B, 7]. Append unlabeled Class_C placeholder.
        b = feature_values.size(0)
        class_placeholder = torch.zeros(b, 1, device=feature_values.device)
        node_values = torch.cat([feature_values, class_placeholder], dim=1).unsqueeze(-1)
        x = self.pre(node_values.reshape(-1,1)).reshape(b, 8, -1)

        h1 = self.gcn1(x, adj_norm)
        h2 = self.gcn2(h1, adj_norm)
        h3 = self.gcn3(h2, adj_norm)

        # ConvLSTM sequence = successive GCN-depth embeddings
        seq = torch.stack([h1, h2, h3], dim=1)  # [B, 3, nodes, hidden]
        seq = seq.permute(0, 1, 3, 2).contiguous()  # [B, 3, hidden, nodes]
        fused = self.fusion(seq)  # [B, hidden, nodes]
        class_embedding = fused[:, :, -1]  # Class_C node
        return self.post(class_embedding)

model = LesionGraphNet().to(DEVICE)
A_tensor = torch.tensor(A_norm, dtype=torch.float32, device=DEVICE)
print(model)

In [ ]:
# 9. DataLoaders and training
class FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(FeatureDataset(X_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(FeatureDataset(X_val, y_val),
                        batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

def evaluate_loader(model, loader):
    model.eval()
    losses, all_y, all_prob = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb, A_tensor)
            losses.append(criterion(logits, yb).item())
            all_y.extend(yb.cpu().numpy())
            all_prob.extend(torch.softmax(logits, dim=1)[:,1].cpu().numpy())
    pred = (np.array(all_prob) >= 0.5).astype(int)
    return (float(np.mean(losses)), accuracy_score(all_y, pred),
            np.array(all_y), np.array(all_prob))

best_val = float("inf")
best_state = None
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb, A_tensor)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)

    train_loss = running_loss / max(total,1)
    train_acc = correct / max(total,1)
    val_loss, val_acc, _, _ = evaluate_loader(model, val_loader)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

    if (epoch+1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d}/{EPOCHS} | "
              f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
              f"val loss {val_loss:.4f} acc {val_acc:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(DEVICE)

fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(history["train_loss"], label="Train"); ax[0].plot(history["val_loss"], label="Validation")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history["train_acc"], label="Train"); ax[1].plot(history["val_acc"], label="Validation")
ax[1].set_title("Accuracy"); ax[1].legend()
plt.show()

In [ ]:
# 10. Final test evaluation
test_loader = DataLoader(FeatureDataset(X_test, y_test),
                         batch_size=BATCH_SIZE, shuffle=False)
test_loss, test_acc, y_true, y_prob = evaluate_loader(model, test_loader)
y_pred = (y_prob >= 0.5).astype(int)

print("Test loss:", round(test_loss, 4))
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall/Sensitivity:", recall_score(y_true, y_pred, zero_division=0))
print("F1:", f1_score(y_true, y_pred, zero_division=0))
print("MCC:", matthews_corrcoef(y_true, y_pred))
print("Specificity:", confusion_matrix(y_true, y_pred).ravel()[0] /
      max(confusion_matrix(y_true, y_pred).ravel()[0] +
          confusion_matrix(y_true, y_pred).ravel()[1], 1))
print("ROC-AUC:", roc_auc_score(y_true, y_prob))
print(classification_report(y_true, y_pred, target_names=["Benign","Malignant"], zero_division=0))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Benign","Malignant"], yticklabels=["Benign","Malignant"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Test confusion matrix"); plt.show()

fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.plot(fpr, tpr, label=f"AUC={auc(fpr,tpr):.4f}")
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC curve"); plt.legend(); plt.show()

## 11. Save model and preprocessing objects

Keep the feature order and scalers with the model. For a real external validation, apply the same fixed preprocessing, selected feature order, scaler and training-derived graph. Do not refit them on PH2/DermNet or test data.

In [ ]:
import joblib
OUTPUT_DIR = "/content/lesiongraphnet_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "adjacency_normalized": A_norm,
    "selected_features": SELECTED_FEATURES,
    "feature_names_all": FEATURE_NAMES,
    "seed": SEED
}, os.path.join(OUTPUT_DIR, "lesiongraphnet.pt"))

joblib.dump(feature_scaler, os.path.join(OUTPUT_DIR, "feature_scaler.joblib"))
joblib.dump(minmax_all, os.path.join(OUTPUT_DIR, "minmax_scaler_all_features.joblib"))
print("Saved to:", OUTPUT_DIR)

## 12. Research reproduction checklist / limitations

1. Recreate the paper's exact 3,297-image subset (including lesion-ID deduplication and near-duplicate handling) before comparing reported scores.
2. Verify segmentation masks manually or against expert masks; K-means can select the wrong dark region.
3. Keep all feature selection, scaling and Pearson graph estimation inside each training fold during cross-validation.
4. The paper describes both a dataset-level feature-correlation graph and an 8-node per-image graph with `Class_C`. This notebook implements the latter inference description and uses training-set correlations for the shared topology.
5. The paper's reproducibility table says 3 GCN layers, hidden size 64, ConvLSTM1D, SGD 0.01, dropout 0.3, batch 64, 250 epochs and seed 42. Some FFN/ConvLSTM internals are not fully specified; this notebook's explicit implementation is therefore paper-guided, not bit-for-bit source reproduction.
6. Report your own results with confidence intervals and repeated runs. Do not copy the paper's 98.62% result as your result.
7. This model is for research/education and must not be used to diagnose people or replace dermatologist assessment.

## 13. Optional 5-fold cross-validation (paper's additional validation)

This cell is computationally expensive because it retrains the network for every fold. It rebuilds the scaler and Pearson graph from each fold's training partition only, avoiding information leakage. Run after the main implementation cells.

In [ ]:
# Optional 5-fold CV. Each fold gets its own scaler, graph and fresh model.
def run_5fold_cv(features_dataframe, epochs=EPOCHS):
    X_raw = features_dataframe[FEATURE_NAMES].values.astype(np.float32)
    y_all = features_dataframe["label"].values.astype(np.int64)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_results = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_raw, y_all), start=1):
        print(f"\\n========== Fold {fold}/5 ==========")
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_raw[tr_idx][:, selected_idx]).astype(np.float32)
        Xva = scaler.transform(X_raw[va_idx][:, selected_idx]).astype(np.float32)
        ytr, yva = y_all[tr_idx], y_all[va_idx]

        _, _, _, fold_adj = build_adjacency(Xtr, threshold=EDGE_THRESHOLD)
        fold_adj_t = torch.tensor(fold_adj, dtype=torch.float32, device=DEVICE)

        tr_loader = DataLoader(FeatureDataset(Xtr, ytr), batch_size=BATCH_SIZE,
                               shuffle=True, drop_last=False)
        va_loader = DataLoader(FeatureDataset(Xva, yva), batch_size=BATCH_SIZE,
                               shuffle=False)

        fold_model = LesionGraphNet().to(DEVICE)
        fold_optimizer = torch.optim.SGD(fold_model.parameters(), lr=LR)
        fold_best_loss, fold_best_state = float("inf"), None

        for ep in range(epochs):
            fold_model.train()
            for xb, yb in tr_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                fold_optimizer.zero_grad()
                loss = criterion(fold_model(xb, fold_adj_t), yb)
                loss.backward()
                fold_optimizer.step()

            fold_model.eval()
            losses, true, probs = [], [], []
            with torch.no_grad():
                for xb, yb in va_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    logits = fold_model(xb, fold_adj_t)
                    losses.append(criterion(logits, yb).item())
                    true.extend(yb.numpy())
                    probs.extend(torch.softmax(logits, dim=1)[:,1].cpu().numpy())

            mean_loss = float(np.mean(losses))
            if mean_loss < fold_best_loss:
                fold_best_loss = mean_loss
                fold_best_state = {k:v.detach().cpu().clone()
                                   for k,v in fold_model.state_dict().items()}

        fold_model.load_state_dict(fold_best_state)
        fold_model.to(DEVICE)
        pred = (np.asarray(probs) >= 0.5).astype(int)
        result = {
            "fold": fold,
            "accuracy": accuracy_score(true, pred),
            "precision": precision_score(true, pred, zero_division=0),
            "recall": recall_score(true, pred, zero_division=0),
            "f1": f1_score(true, pred, zero_division=0),
            "auc": roc_auc_score(true, probs) if len(np.unique(true)) == 2 else np.nan,
            "mcc": matthews_corrcoef(true, pred)
        }
        fold_results.append(result)
        print(result)

    results_df = pd.DataFrame(fold_results)
    print("\\nMean ± standard deviation:")
    display(results_df.drop(columns="fold").agg(["mean", "std"]))
    return results_df

# Uncomment to run:
# cv_results = run_5fold_cv(features_df, epochs=EPOCHS)